# H₂ dynamics animation

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
# ===== 第一步：使用 OpenFermion 构造 H2 Toy 模型的费米子哈密顿量 =====
from openfermion.ops import FermionOperator
from openfermion.transforms import bravyi_kitaev
from openfermion.utils import hermitian_conjugated
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
# 构造简化 H2 分子的 toy Hamiltonian
H_f = FermionOperator('', -0.8126)
H_f += FermionOperator('0^ 0', -0.0458)
H_f += FermionOperator('1^ 1', -0.0458)
H_f += FermionOperator('2^ 2', 0.1743)
H_f += FermionOperator('3^ 3', 0.1743)
H_f += FermionOperator('0^ 1^ 1 0', 0.1220)
H_f += FermionOperator('2^ 3^ 3 2', 0.1676)
H_f += FermionOperator('0^ 2^ 2 0', 0.1686)
H_f += FermionOperator('1^ 3^ 3 1', 0.1686)
H_f += FermionOperator('0^ 3^ 3 0', 0.1202)
H_f += FermionOperator('1^ 2^ 2 1', 0.1202)
H_f += FermionOperator('0^ 2', -0.0450)
H_f += FermionOperator('1^ 3', -0.0450)
H_f += hermitian_conjugated(H_f)

In [ ]:
# ===== 第二步：使用 BK 变换得到泡利量子比特哈密顿量 =====
H_q = bravyi_kitaev(H_f)

# 工具函数：将 QubitOperator 转换为 QuTiP Hamiltonian
n_qubits = 4
def qubit_operator_to_qutip(qubit_op, n_qubits):
    pauli_map = {'X': sigmax(), 'Y': sigmay(), 'Z': sigmaz()}
    H = 0
    for term, coeff in qubit_op.terms.items():
        ops = [qeye(2)] * n_qubits
        for idx, op in term:
            ops[idx] = pauli_map[op]
        H += coeff * tensor(ops)
    return H

In [ ]:
# ===== 第三步：构造两个玻色子振动模 =====
n_vib = 5
w1, w2 = 1.0, 1.2
J = 0.1
a1 = destroy(n_vib)
a2 = destroy(n_vib)
H_vib = (
    w1 * tensor(a1.dag()*a1, qeye(n_vib)) +
    w2 * tensor(qeye(n_vib), a2.dag()*a2) +
    J * (tensor(a1.dag(), a2) + tensor(a1, a2.dag()))
)

In [ ]:
# ===== 第四步：构造电子-振动模耦合项 =====
sigma_z = tensor(sigmaz(), qeye(2), qeye(2), qeye(2))
H_couple = tensor(sigma_z, tensor(a1 + a1.dag(), qeye(n_vib))) * 0.2

# ===== 第五步：构造总哈密顿量 H_tot =====
I_q = tensor([qeye(2) for _ in range(n_qubits)])
I_b = qeye(n_vib)
I_bb = tensor(I_b, I_b)

H_qubit = tensor(qubit_operator_to_qutip(H_q, n_qubits), I_bb)
H_tot = H_qubit + tensor(I_q, H_vib) + H_couple


In [ ]:
# ===== 第六步：初始化系统初态 psi0 =====
psi_qubit = tensor([basis(2, 0) for _ in range(n_qubits)])
psi_boson = tensor(basis(n_vib, 1), basis(n_vib, 0))
psi0 = tensor(psi_qubit, psi_boson)

# ===== 第七步：设置时间与观测量，运行多个热耗散条件下的模拟（包含电子耗散） =====
tlist = np.linspace(0, 50, 100)
n1 = tensor(I_q, tensor(a1.dag() * a1, qeye(n_vib)))
n2 = tensor(I_q, tensor(qeye(n_vib), a2.dag() * a2))

results = {}
for n_th in [0.0, 0.5, 1.5, 3.0]:
    gamma1, gamma2 = 0.02, 0.02
    c_ops = [
        np.sqrt(gamma1 * (n_th + 1)) * tensor(I_q, tensor(a1, I_b)),
        np.sqrt(gamma1 * n_th)       * tensor(I_q, tensor(a1.dag(), I_b)),
        np.sqrt(gamma2 * (n_th + 1)) * tensor(I_q, tensor(I_b, a2)),
        np.sqrt(gamma2 * n_th)       * tensor(I_q, tensor(I_b, a2.dag())),
        np.sqrt(0.01) * tensor(sigmax(), qeye(2), qeye(2), qeye(2), I_bb),
    ]
    res = mesolve(H_tot, psi0, tlist, c_ops, [n1, n2])
    results[n_th] = res.expect

In [ ]:
# ===== 动图可视化不同热噪声条件下的能量转移（3D） =====
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.set_xlim(0, 50)
ax.set_ylim(0, max(results.keys()))
ax.set_zlim(0, 2)
ax.set_xlabel("Time")
ax.set_ylabel("n_th (thermal excitation)")
ax.set_zlabel("Occupation")
ax.set_title("H₂ Vibrational Mode Energy Distribution with Thermal Dissipation")

x_all = np.array([])
y_all = np.array([])
z1_all = np.array([])
z2_all = np.array([])

for n_th in results:
    z1 = results[n_th][0]
    z2 = results[n_th][1]
    x_all = np.concatenate((x_all, tlist))
    y_all = np.concatenate((y_all, np.full_like(tlist, n_th)))
    z1_all = np.concatenate((z1_all, z1))
    z2_all = np.concatenate((z2_all, z2))

ax.plot(x_all, y_all, z1_all, color='blue', label='a₁')
ax.plot(x_all, y_all, z2_all, color='red', label='a₂')
ax.legend()
plt.tight_layout()
plt.show()
